# 📊 Tahap 2: Case Representation
Ekstraksi metadata dan fitur teks dari setiap putusan → simpan ke `cases.csv` & `cases.json`

> **Prasyarat:** Tahap 1 sudah selesai (ada file `.txt` di `data/raw/`)

In [ ]:
# Cell 1 — Import Library
import os, re, json, glob
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
print("✅ Library siap")

In [ ]:
# Cell 2 — Konfigurasi Path
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)
print(f"✅ Path siap")
print(f"   RAW_DIR  : {RAW_DIR}")
print(f"   PROC_DIR : {PROC_DIR}")
txt_files = glob.glob(os.path.join(RAW_DIR, "*.txt"))
print(f"   TXT ditemukan: {len(txt_files)} file")

## 🔍 Fungsi Ekstraksi Metadata

In [ ]:
# Cell 3 — Fungsi ekstraksi metadata
def extract_no_perkara(text):
    for pat in [
        r"nomor\s*:?\s*([\d]+\s*/\s*pid\.sus\s*/\s*\d{4}(?:/[a-z]+\.?\w*)*)",
        r"nomor\s*:?\s*([\d]+\s*/\s*pid\s*/\s*\d{4}(?:/[a-z]+\.?\w*)*)",
        r"putusan\s+nomor\s*([\w/\.\-]+)",
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m: return m.group(1).strip().upper()
    return "TIDAK_DITEMUKAN"

def extract_tanggal(text):
    bln = {"januari":"01","februari":"02","maret":"03","april":"04","mei":"05",
           "juni":"06","juli":"07","agustus":"08","september":"09",
           "oktober":"10","november":"11","desember":"12"}
    m = re.search(r"(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+(\d{4})", text, re.IGNORECASE)
    if m: return f"{m.group(3)}-{bln[m.group(2).lower()]}-{int(m.group(1)):02d}"
    return "TIDAK_DITEMUKAN"

def extract_pasal(text):
    found = set()
    for pat in [
        r"(pasal\s+\d+(?:\s+ayat\s+\(\d+\))?\s+(?:undang-undang|uu)\s+(?:nomor\s+)?\d+\s+tahun\s+\d+)",
        r"(pasal\s+\d+(?:\s+ayat\s+\(\d+\))?(?:\s+jo\.?\s*pasal\s+\d+(?:\s+ayat\s+\(\d+\))?)*)",
    ]:
        for m in re.findall(pat, text, re.IGNORECASE)[:3]:
            found.add(m.strip())
    return " | ".join(list(found)[:3]) if found else "TIDAK_DITEMUKAN"

def extract_terdakwa(text):
    for pat in [
        r"terdakwa\s*:?\s*([A-Z][A-Z\s\.,]+?)(?:\n|;|,\s*(?:bin|binti|alias))",
        r"nama\s*lengkap\s*:?\s*([A-Z][A-Z\s\.,]+?)(?:\n|;)",
        r"(?:terdakwa|terpidana)\s+([A-Z][A-Z\s]+?)(?:\s+(?:bin|binti|alias|terbukti))",
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            n = m.group(1).strip()
            if 3 < len(n) < 100: return n
    return "TIDAK_DITEMUKAN"

def extract_amar_putusan(text):
    for pat in [
        r"(mengadili\s*:.*?)(?=\n\n|\Z)",
        r"(amar\s+putusan\s*:.*?)(?=\n\n|\Z)",
        r"((?:menghukum|menjatuhkan|memidana).*?)(?=\n\n|\Z)",
    ]:
        m = re.search(pat, text, re.IGNORECASE|re.DOTALL)
        if m: return m.group(1).strip()[:1000]
    return "TIDAK_DITEMUKAN"

def extract_ringkasan_fakta(text):
    for pat in [
        r"(?:bahwa\s+terdakwa.*?)(?=menimbang|mengadili|\Z)",
        r"(?:dakwaan\s*:.*?)(?=tuntutan|mengadili|\Z)",
    ]:
        m = re.search(pat, text, re.IGNORECASE|re.DOTALL)
        if m: return m.group(0).strip()[:1000]
    p = [x.strip() for x in text.split("\n\n") if len(x.strip())>50]
    return " ".join(p[:3])[:800]

def extract_pidana_penjara(text):
    for pat in [
        r"pidana\s+penjara\s+selama\s+([\d]+\s+(?:tahun|bulan|hari)(?:\s+[\d]+\s+(?:bulan|hari))?)",
        r"penjara\s+(?:selama\s+)?([\d]+\s+(?:tahun|bulan))",
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m: return m.group(1).strip()
    return "TIDAK_DITEMUKAN"

def extract_uang_pengganti(text):
    for pat in [
        r"uang\s+pengganti\s+(?:sebesar\s+)?(?:rp\.?\s*)?([\d\.,]+(?:\s+(?:miliar|juta|ribu))?)",
        r"membayar\s+uang\s+pengganti\s+(?:rp\.?\s*)?([\d\.,]+(?:\s+(?:miliar|juta|ribu))?)",
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m: return "Rp " + m.group(1).strip()
    return "TIDAK_DITEMUKAN"

def label_putusan(amar):
    a = amar.lower()
    if any(k in a for k in ["menyatakan terdakwa terbukti","menghukum","memidana","pidana penjara"]):
        return "bersalah"
    elif any(k in a for k in ["membebaskan","bebas dari dakwaan","tidak terbukti"]):
        return "bebas"
    return "bersalah"

print("✅ Semua fungsi ekstraksi siap")

## ▶️ Jalankan Pipeline Representasi

In [ ]:
# Cell 4 — Pipeline utama
txt_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.txt")))
if not txt_files:
    print("❌ Tidak ada file .txt! Jalankan dulu Tahap 1.")
else:
    print(f"{'='*55}")
    print(f"  TAHAP 2: CASE REPRESENTATION")
    print(f"  Proses {len(txt_files)} file teks...")
    print(f"{'='*55}")
    records = []
    for idx, fpath in enumerate(txt_files, 1):
        case_id = os.path.splitext(os.path.basename(fpath))[0]
        print(f"  [{idx:02d}/{len(txt_files)}] {case_id}", end=" → ")
        with open(fpath, "r", encoding="utf-8") as f:
            text = f.read()
        amar  = extract_amar_putusan(text)
        label = label_putusan(amar)
        wc    = len(text.split())
        records.append({
            "case_id"         : case_id,
            "no_perkara"      : extract_no_perkara(text),
            "tanggal"         : extract_tanggal(text),
            "jenis_perkara"   : "Pidana Khusus - Korupsi Kerugian Keuangan Negara",
            "pasal"           : extract_pasal(text),
            "terdakwa"        : extract_terdakwa(text),
            "ringkasan_fakta" : extract_ringkasan_fakta(text),
            "amar_putusan"    : amar,
            "pidana_penjara"  : extract_pidana_penjara(text),
            "uang_pengganti"  : extract_uang_pengganti(text),
            "jumlah_kata"     : wc,
            "label"           : label,
            "text_full"       : text,
        })
        print(f"✅ label={label} | {wc:,} kata")
    df = pd.DataFrame(records)

In [ ]:
# Cell 5 — Simpan ke CSV dan JSON
csv_path  = os.path.join(PROC_DIR, "cases.csv")
json_path = os.path.join(PROC_DIR, "cases.json")
full_path = os.path.join(PROC_DIR, "cases_full.csv")

df.drop(columns=["text_full"]).to_csv(csv_path, index=False, encoding="utf-8-sig")
df.drop(columns=["text_full"]).to_json(json_path, orient="records", force_ascii=False, indent=2)
df.to_csv(full_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Dataset tersimpan:")
print(f"   cases.csv      : {csv_path}")
print(f"   cases.json     : {json_path}")
print(f"   cases_full.csv : {full_path}")
print(f"\n📊 Statistik:")
print(f"   Total kasus        : {len(df)}")
print(f"   Distribusi label   :\n{df['label'].value_counts().to_string()}")
print(f"   Rata-rata kata     : {df['jumlah_kata'].mean():.0f}")

## 👁️ Pratinjau Dataset

In [ ]:
# Cell 6 — Tampilkan sample data
cols = ["case_id","no_perkara","tanggal","terdakwa","pidana_penjara","label"]
print("📋 5 baris pertama:")
print(df[cols].head().to_string(index=False))